# Example - Combinatorial Potentials

## Setup

Run all cells from top to bottom. Package revision: **990f446**.

In [1]:
from importlib.metadata import PackageNotFoundError, distribution
import json
import subprocess
import sys

source = "https://github.com/magin67/polyform_algebra/archive/990f4462b17a1afaad25666220ffc1fbc2cd4d26.zip"
try:
    installed = distribution("polyform_algebra")
    installed_url = json.loads(installed.read_text("direct_url.json") or "{}").get("url")
except PackageNotFoundError:
    installed = None
    installed_url = None

# A tag and a development revision can report the same version string.
if installed_url != source:
    if "polyform_algebra" in sys.modules:
        raise RuntimeError("Restart the kernel before installing the pinned package revision.")
    command = [sys.executable, "-m", "pip", "install", "-q"]
    if installed is not None and installed.version == "0.9.0":
        command += ["--force-reinstall", "--no-deps"]
    subprocess.check_call(command + [source])


In [2]:
import polyform_algebra
assert polyform_algebra.get_version() == "0.9.0"
print("polyform_algebra", polyform_algebra.get_version())

polyform_algebra 0.9.0


Compute combinatorial potentials for prescribed types and check them by enumeration. The forest examples also use partition forms, contraction, and exact spanning-tree coefficients. These calculations verify selected examples. The series notes give the general proofs.


Run the fragments from top to bottom. The notebook installs the pinned package revision. [polyform-algebra 990f446](https://github.com/magin67/polyform_algebra/commit/990f4462b17a1afaad25666220ffc1fbc2cd4d26).


## Partitions and the second-kind potential

Enumerate the three partitions of $\{a,b,c,d\}$ into two pairs and compare the count with $s_2([2,2])$. For sets of up to five elements, check the formula for each type. Summing over all types gives the Bell number. Summing over types with a fixed block count gives the Stirling number of the second kind.

[Related section of the note](https://magin67.github.io/pmg-notes/en/polyform-algebra-and-combinatorics/combinatorial-potentials/partitions-and-the-combinatorial-potential-of-the-second-kind#one-type-and-three-partitions).

In [3]:
from collections import Counter
from itertools import combinations, permutations
from math import comb, factorial, perm, prod

from sympy.functions.combinatorial.numbers import bell, stirling
from sympy.utilities.iterables import multiset_partitions

from polyform_algebra.combinatorics import (
    first_kind_potential, path_cycle_potential, second_kind_potential,
)


def block_sizes(blocks):
    return tuple(sorted((len(block) for block in blocks), reverse=True))


four = tuple("abcd")
pairs = [blocks for blocks in multiset_partitions(four, 2)
         if block_sizes(blocks) == (2, 2)]
assert len(pairs) == second_kind_potential((2, 2)) == 3
print("s₂([2,2]) =", len(pairs), "|", [" | ".join(map("".join, p)) for p in pairs])

for n in range(1, 6):
    labels = tuple("abcde"[:n])
    counts = Counter(block_sizes(blocks) for blocks in multiset_partitions(labels))
    assert all(count == second_kind_potential(sizes) for sizes, count in counts.items())
    assert sum(counts.values()) == bell(n)
    for m in range(1, n + 1):
        assert sum(count for sizes, count in counts.items() if len(sizes) == m) == stirling(n, m)
print("B₅ =", bell(5), "| S(5,2) =", stirling(5, 2))

s₂([2,2]) = 3 | ['ab | cd', 'ac | bd', 'ad | bc']
B₅ = 52 | S(5,2) = 15


## From partitions to forests on five vertices

The entries of type $[2,1]$ are grades. To count partitions of the vertices, use block sizes $[3,2]$. The potential $s_2([2,1])=3$ counts partitions of an auxiliary three-element set. On five vertices there are ten partitions and thirty forests. On the left, sum the products of edge forms for all these forests. On the right, sum the partition forms and multiply by $3$. Compare the coefficients exactly.

[Related section of the note](https://magin67.github.io/pmg-notes/en/polyform-algebra-and-combinatorics/combinatorial-potentials/combinatorial-potentials-in-symmetric-polyforms#why-there-are-ten-partitions-but-thirty-forests).

In [4]:
from functools import reduce

from polyform_algebra.core import (
    Context, Network, contract, equivalent, form_product, network,
    partition_form, spanning_tree_weight,
)


def partition(context, blocks=()):
    return partition_form(context, blocks).value


def complete_graph(labels):
    return network(labels, tuple((a, b, 1) for a, b in combinations(labels, 2)))


def product_of_edges(context, edges):
    return reduce(
        lambda form, edge: form_product(form, partition(context, (edge,))).value,
        edges, partition(context),
    )


five = tuple("abcde")
context = Context(five)
zero = 0 * partition(context)
type_partitions = [
    tuple(map(tuple, blocks)) for blocks in multiset_partitions(five, 2)
    if block_sizes(blocks) == (3, 2)
]
E5_21 = sum((partition(context, blocks) for blocks in type_partitions), start=zero)

forest_edges = []
for edges in combinations(combinations(five, 2), 3):
    forest = Network(context, tuple((a, b, 1) for a, b in edges))
    if block_sizes(forest.components) == (3, 2):
        forest_edges.append(edges)
forest_sum = sum((product_of_edges(context, edges) for edges in forest_edges), start=zero)

assert second_kind_potential((2, 1)) == 3
assert len(type_partitions) == second_kind_potential((3, 2)) == 10
assert len(forest_edges) == 3 * len(type_partitions) == 30
assert equivalent(forest_sum, 3 * E5_21).value
print("s₂([2,1]) = 3 | #E₅[2,1] = 10 | 30 forest terms = 3 E₅[2,1]")

s₂([2,1]) = 3 | #E₅[2,1] = 10 | 30 forest terms = 3 E₅[2,1]


## Joining prescribed components

Keep the prescribed edges $ab$ or $ab,cd$ and enumerate the pairs of new edges that complete the forest to a tree. Count adjacent and disjoint new-edge pairs separately. Compare the total with the Cayley–Moon formula and with the spanning-tree coefficient of the complete graph after contracting the prescribed blocks.

[Related section of the note](https://magin67.github.io/pmg-notes/en/polyform-algebra-and-combinatorics/combinatorial-potentials/cayley%27s-theorem-and-joining-prescribed-components#contraction-to-a-weighted-graph).

In [5]:
def completions(labels, groups):
    complete = complete_graph(labels)
    contracted = contract(complete, groups).value.network
    sizes = tuple(map(len, groups.values()))
    expected = prod(sizes) * len(labels) ** (len(sizes) - 2)
    assert spanning_tree_weight(contracted).value == expected

    # In both examples each original component is a singleton or a fixed edge.
    prescribed = tuple(tuple(block) for block in groups.values() if len(block) == 2)
    candidates = [edge for edge in combinations(labels, 2) if edge not in prescribed]
    distribution = Counter()
    for added in combinations(candidates, len(sizes) - 1):
        tree = Network(complete.context, tuple((a, b, 1) for a, b in prescribed + added))
        if len(tree.components) == 1:
            new_edges = Network(complete.context, tuple((a, b, 1) for a, b in added))
            positive_grades = tuple(size - 1 for size in block_sizes(new_edges.components) if size > 1)
            distribution[positive_grades] += 1
    assert sum(distribution.values()) == expected
    return expected, (distribution[(2,)], distribution[(1, 1)])


assert completions(four, {"A": ("a", "b"), "B": ("c",), "C": ("d",)}) == (8, (6, 2))
assert completions(five, {"A": ("a", "b"), "B": ("c", "d"), "C": ("e",)}) == (20, (12, 8))
print("T([2,1,1]) = 6 + 2 = 8 | T([2,2,1]) = 12 + 8 = 20")

T([2,1,1]) = 6 + 2 = 8 | T([2,2,1]) = 12 + 8 = 20


## Forest coefficients and Stirling numbers

For each $1\leqslant k\leqslant6$, compute $\operatorname{St}(k,m)$ in three ways. First sum over ordinary block sizes, using Cayley's factor for each block with its added root. Then build a row of $r$-Stirling numbers from $r=k+1$ separate markers, and separately compute a finite difference of the Abel polynomial. Display the rows for $k=3$ and $k=4$.

[Related section of the note](https://magin67.github.io/pmg-notes/en/polyform-algebra-and-combinatorics/combinatorial-potentials/falling-factorials,-rooted-forests,-and-stirling-numbers#why-forest-coefficients-are-r-stirling-numbers).

In [6]:
def restricted_stirling(markers, positions):
    # Start with the markers in separate blocks. Each new position joins
    # one of the existing blocks or forms a new block of its own.
    row = {markers: 1}
    for _ in range(positions):
        next_row = Counter()
        for blocks, count in row.items():
            next_row[blocks] += blocks * count
            next_row[blocks + 1] += count
        row = next_row
    return row


for k in range(1, 7):
    labels = tuple("abcdef"[:k])
    types = {block_sizes(blocks) for blocks in multiset_partitions(labels)}
    row = restricted_stirling(markers=k + 1, positions=k - 1)
    coefficients = []
    for m in range(1, k + 1):
        by_type = sum(second_kind_potential(sizes) * prod((size + 1) ** (size - 1) for size in sizes)
                      for sizes in types if len(sizes) == m)
        difference = sum((-1) ** (m-i) * comb(m, i) * i * (i + k) ** (k - 1)
                         for i in range(m + 1))
        assert by_type == row[k + m]
        assert factorial(m) * by_type == difference
        coefficients.append(by_type)
    if k in (3, 4):
        print(f"St({k}, m) =", tuple(coefficients))
        assert tuple(coefficients) == {3: (16, 9, 1), 4: (125, 91, 18, 1)}[k]

St(3, m) = (16, 9, 1)
St(4, m) = (125, 91, 18, 1)


## Cycles and the first-kind potential

Enumerate the permutations of four letters and group them by cycle lengths. Check the count against $s_1$ and against $s_2$ multiplied by the number of cyclic orders within the blocks. On five letters, count the permutations with no fixed points separately.

[Related section of the note](https://magin67.github.io/pmg-notes/en/polyform-algebra-and-combinatorics/combinatorial-potentials/cyclic-partitions-and-the-combinatorial-potential-of-the-first-kind#the-potential-of-a-cycle-type).

In [7]:
def cycle_type(labels, images):
    permutation = dict(zip(labels, images))
    remaining = set(labels)
    lengths = []
    while remaining:
        vertex = min(remaining)
        length = 0
        while vertex in remaining:
            remaining.remove(vertex)
            vertex = permutation[vertex]
            length += 1
        lengths.append(length)
    return tuple(sorted(lengths, reverse=True))


types4 = Counter(cycle_type(four, images) for images in permutations(four))
for sizes, count in sorted(types4.items(), reverse=True):
    assert count == first_kind_potential(sizes)
    assert count == second_kind_potential(sizes) * prod(factorial(size - 1) for size in sizes)
    print(f"s₁{list(sizes)} =", count)
assert sum(types4.values()) == factorial(len(four))

derangements5 = sum(all(a != b for a, b in zip(five, images)) for images in permutations(five))
assert derangements5 == first_kind_potential((5,)) + first_kind_potential((3, 2)) == 44
print("D₅ =", derangements5)

s₁[4] = 6
s₁[3, 1] = 8
s₁[2, 2] = 3
s₁[2, 1, 1] = 6
s₁[1, 1, 1, 1] = 1
D₅ = 44


## Paths, cycles, and construction histories

First check seven small state types. For a two-cycle and a separate open arc, enumerate how the three arc labels can be assigned to components. Then on vertices $a,b,c$, enumerate directed two-arc paths and every order of adding their arcs. The checks distinguish the potential $u=2$, the graph count $N_3=6$, and the history count $H_3=12$.

[Related section of the note](https://magin67.github.io/pmg-notes/en/polyform-algebra-and-combinatorics/combinatorial-potentials/the-potential-of-path-and-cycle-states#graphs-on-prescribed-vertices-and-construction-histories).

In [8]:
examples = (
    ((1, 1), (), 1), ((1,), (1,), 2), ((2,), (), 1), ((), (2,), 2),
    ((), (1, 1), 1), ((2,), (1,), 3), ((1,), (2,), 6),
)
assert all(path_cycle_potential(cycles, paths) == count for cycles, paths, count in examples)
arc_labels = tuple("ijk")
mixed_arc_types = {
    (tuple(sorted(cycle)), path) for *cycle, path in permutations(arc_labels)
}
assert len(mixed_arc_types) == path_cycle_potential((2,), (1,)) == 3

vertices = tuple("abc")
directed_paths = {((a, b), (b, c)) for a, b, c in permutations(vertices)}
histories = {history for edges in directed_paths for history in permutations(edges)}
u = path_cycle_potential(path_lengths=(2,))
assert len(histories) == u * perm(len(vertices), 3) == 12
assert len(directed_paths) == len(histories) // factorial(2) == 6
print("u([2, open 1]) = 3 | two-arc path: u = 2, N₃ = 6, H₃ = 12")

u([2, open 1]) = 3 | two-arc path: u = 2, N₃ = 6, H₃ = 12
